# Clinical ASR Benchmark: Colab quickstart

Runtime > Change runtime type > **T4 GPU**, then run the cells in order.
Transcripts are cached in `results/transcripts/`, so if Colab disconnects you can rerun and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/PeidaWu/clinical-asr-bench.git
%cd clinical-asr-bench
!pip install -q jiwer rouge-score librosa soundfile tabulate

In [ ]:
!python -m pytest -q tests

## 1. Smoke test: 2 consultations, whisper-small, clean audio (~3 min)

In [ ]:
!python scripts/run_benchmark.py --n 2 --models openai/whisper-small --snrs clean --out results_smoke

## 2. Main experiment
10 consultations × 2 models × 4 noise levels × with/without vocabulary prompt.
On a T4 this takes roughly 1–2 hours; start with fewer `--snrs` if you are short on time.

In [ ]:
!python scripts/run_benchmark.py --n 10 \n    --models openai/whisper-small openai/whisper-large-v3-turbo \n    --snrs clean 20 10 5 --bias

In [ ]:
!python scripts/plot_results.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/summary.md').read()))
display(Image('results/wer_vs_snr.png'))

In [ ]:
import pandas as pd
m = pd.read_csv('results/missed_terms.csv')
# Which medical terms does the best model miss most often on clean audio?
m[(m.model.str.contains('large')) & (m.condition == 'clean') & (m.decoding == 'longform')] \n  .groupby('term').missed.sum().sort_values(ascending=False).head(15)

## 3. Transcript → structured note with an open-weight LLM
Compares notes generated from the human transcript (oracle) with notes generated from ASR output.

In [ ]:
!python scripts/run_notes.py --n 10 --asr-tags \n    whisper-large-v3-turbo__clean__longform whisper-large-v3-turbo__babble_snr5__longform
display(Markdown(open('results/notes_summary.md').read()))

In [ ]:
# Inspect one generated note next to the clinician's note
import json, glob
f = sorted(glob.glob('results/notes/*__oracle.json'))[0]
print(json.dumps(json.load(open(f)), indent=2))

## 4. Save results back to GitHub
Download `results/summary.md`, `results/notes_summary.md`, `results/wer_vs_snr.png`, commit them, and paste the tables into the README.

In [ ]:
!zip -r results.zip results/summary.* results/notes_summary.md results/wer_vs_snr.png results/*.csv
from google.colab import files
files.download('results.zip')